# Gold Layer - Confluence Business Aggregations

Reads Silver Parquet files and produces 4 gold tables:
1. `confluence_content_by_space` - metrics per space
2. `confluence_author_activity` - contributions per author
3. `confluence_content_timeline` - daily content creation
4. `confluence_most_discussed` - pages ranked by comments

Writes both Parquet and CSV.

In [ ]:
import os
import pandas as pd
from datetime import datetime, timezone

BASE = '/lakehouse/default/Files'

def read_silver(table):
    path = f'{BASE}/silver/{table}/data.parquet'
    if os.path.exists(path):
        return pd.read_parquet(path)
    return pd.DataFrame()

def write_gold(df, table):
    out_dir = f'{BASE}/gold/{table}'
    os.makedirs(out_dir, exist_ok=True)
    df.to_parquet(f'{out_dir}/data.parquet', index=False)
    df.to_csv(f'{out_dir}/data.csv', index=False)
    print(f'  Wrote {len(df)} rows to gold/{table} (Parquet + CSV)')

pages = read_silver('confluence_pages')
comments = read_silver('confluence_comments')
print(f'Loaded silver data: {len(pages)} pages, {len(comments)} comments')

In [ ]:
# ── Content By Space ─────────────────────────────────────────────
if not pages.empty:
    cbs = (
        pages.groupby('space_key')
        .agg(
            page_count=('page_id', 'nunique'),
            total_words=('word_count', 'sum'),
            avg_word_count=('word_count', 'mean'),
            total_content_length=('content_length', 'sum'),
            latest_update=('last_updated_date', 'max'),
        )
        .reset_index()
        .sort_values('page_count', ascending=False)
    )
    cbs['avg_word_count'] = cbs['avg_word_count'].round(0).astype(int)
    cbs['_aggregated_at'] = datetime.now(timezone.utc).isoformat()
    write_gold(cbs, 'confluence_content_by_space')
else:
    print('No pages data — skipping content_by_space.')

In [ ]:
# ── Author Activity ──────────────────────────────────────────────
if not pages.empty:
    page_agg = (
        pages.groupby('created_by')
        .agg(pages_created=('page_id', 'nunique'), pages_total_words=('word_count', 'sum'))
        .reset_index()
        .rename(columns={'created_by': 'author'})
    )
    if not comments.empty:
        comment_agg = (
            comments.groupby('author')
            .agg(comments_made=('comment_id', 'nunique'), comments_total_words=('word_count', 'sum'))
            .reset_index()
        )
        agg = page_agg.merge(comment_agg, on='author', how='outer').fillna(0)
    else:
        agg = page_agg.copy()
        agg['comments_made'] = 0
        agg['comments_total_words'] = 0
    for col in ['pages_created', 'pages_total_words', 'comments_made', 'comments_total_words']:
        agg[col] = agg[col].astype(int)
    agg['total_contributions'] = agg['pages_created'] + agg['comments_made']
    agg = agg.sort_values('total_contributions', ascending=False)
    agg['_aggregated_at'] = datetime.now(timezone.utc).isoformat()
    write_gold(agg, 'confluence_author_activity')
else:
    print('No pages data — skipping author_activity.')

In [ ]:
# ── Content Timeline ─────────────────────────────────────────────
rows = []
if not pages.empty and 'created_date' in pages.columns:
    p = pages.copy()
    p['day'] = p['created_date'].dt.date
    rows.append(p.groupby('day').agg(pages_created=('page_id', 'nunique')).reset_index())

if not comments.empty and 'created_date' in comments.columns:
    c = comments.copy()
    c['day'] = c['created_date'].dt.date
    rows.append(c.groupby('day').agg(comments_created=('comment_id', 'nunique')).reset_index())

if rows:
    if len(rows) == 2:
        tl = rows[0].merge(rows[1], on='day', how='outer').fillna(0)
    else:
        tl = rows[0]
        if 'pages_created' not in tl.columns: tl['pages_created'] = 0
        if 'comments_created' not in tl.columns: tl['comments_created'] = 0
    tl = tl.sort_values('day')
    for col in ['pages_created', 'comments_created']:
        if col in tl.columns: tl[col] = tl[col].astype(int)
    tl['total_activity'] = tl.get('pages_created', 0) + tl.get('comments_created', 0)
    tl['cumulative_pages'] = tl.get('pages_created', pd.Series([0])).cumsum()
    tl['_aggregated_at'] = datetime.now(timezone.utc).isoformat()
    write_gold(tl, 'confluence_content_timeline')
else:
    print('No timeline data available.')

In [ ]:
# ── Most Discussed Pages ────────────────────────────────────────
if not pages.empty:
    if not comments.empty:
        cc = (
            comments.groupby('page_id')
            .agg(comment_count=('comment_id', 'nunique'))
            .reset_index()
        )
        cc['page_id'] = cc['page_id'].astype(str)
        pm = pages.copy()
        pm['page_id'] = pm['page_id'].astype(str)
        md = pm[['page_id', 'title', 'space_key', 'word_count']].merge(
            cc, on='page_id', how='left'
        )
        md['comment_count'] = md['comment_count'].fillna(0).astype(int)
    else:
        md = pages[['page_id', 'title', 'space_key', 'word_count']].copy()
        md['comment_count'] = 0
    md = md.sort_values('comment_count', ascending=False)
    md['_aggregated_at'] = datetime.now(timezone.utc).isoformat()
    write_gold(md, 'confluence_most_discussed')
else:
    print('No pages data — skipping most_discussed.')

print('Gold layer complete.')